In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt
from skimage.metrics import mean_squared_error
import random

# Load the image
image_path = "/content/scmimg.tiff"
img = Image.open(image_path)

# Convert image to NumPy array
img_array = np.array(img)

THRESHOLD = 8

# Column wise unscrambling
def unscramble_columns(img_array):
    num_cols = img_array.shape[1]
    ALLCOL = []
    COLSET = [img_array[:, i] for i in range(num_cols)]
    random.shuffle(COLSET)
    PARENTCOL = COLSET.pop(0)  # Select the first column as parent

    while COLSET:
        COLDIFF = [np.abs(PARENTCOL - col) for col in COLSET]
        COUNT = [np.sum(diff <= THRESHOLD) for diff in COLDIFF]

        best_match_idx = np.argmax(COUNT)
        CHILDCOL = COLSET.pop(best_match_idx)

        PARENTCOL = CHILDCOL
        ALLCOL.append(CHILDCOL)

    return np.column_stack(ALLCOL)

# Row wise unscrambling
def unscramble_rows(img_array):
    num_rows = img_array.shape[0]
    ALLROW = []
    ROWSET = [img_array[i, :] for i in range(num_rows)]
    random.shuffle(ROWSET)
    PARENTROW = ROWSET.pop(0)  # Select the first row as parent

    while ROWSET:
        ROWDIFF = [np.abs(PARENTROW - row) for row in ROWSET]
        COUNT = [np.sum(diff <= THRESHOLD) for diff in ROWDIFF]

        best_match_idx = np.argmax(COUNT)
        CHILDROW = ROWSET.pop(best_match_idx)

        PARENTROW = CHILDROW
        ALLROW.append(CHILDROW)

    return np.vstack(ALLROW)


for i in range(1000):
  # Perform decryption
  col_unscrambled = unscramble_columns(img_array)
  final_image = unscramble_rows(col_unscrambled)

  # Display the results
  fig, ax = plt.subplots(1, 2, figsize=(12, 6))
  ax[0].set_title("Encrypted Image")
  ax[0].imshow(img_array, cmap='gray')
  ax[0].axis("off")
  ax[1].set_title(f"Unscrambled Image {i}")
  ax[1].imshow(final_image, cmap='gray')
  ax[1].axis("off")
  plt.show()


#NOTE:
  #• Step 1 (Reverse XOR) is mathematically EXACT — no key needed.
   # The scrambled image is perfectly recovered (minus 1 row/col).
 #
  #• Step 2 (Unscrambling) is approximate. The row/column ORDER
   # may differ from the original, but the IMAGE CONTENT is revealed.
   # This proves the scheme is broken.


